In [12]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
from joblib import dump





In [13]:

%store -r data
data=data
%store -r preprocessor
preprocessor=preprocessor



In [ ]:
#print(data)

      Brand                  Model  year_of_manufacture  \
0   Hyundai  Nexo Fuel Cell Sports                 2022   
1   Renault      Grand Scenic BLUE                 2020   
2   Hyundai  Nexo Fuel Cell Sports                 2024   
3      Ford                  Focus                 2020   
4   Renault       Kango rapid Blue                 2021   
5      Ford                  Focus                 2028   
6   Renault      Grand Scenic BLUE                 2020   
7      Ford                 Fiesta                 2020   
8   Renault      Grand Scenic BLUE                 2020   
9   Renault      Grand Scenic BLUE                 2018   
10  Renault      Grand Scenic BLUE                 2020   
11  Renault       Kango rapid Blue                 2020   
12  Renault      Grand Scenic BLUE                 2020   
13  Hyundai                 loniq5                 2021   
14  Renault       Kango rapid Blue                 2021   
15  Renault      Grand Scenic BLUE                 2020 

### Define variable

In [14]:
# Prepare the full feature set
# x_content is the feature set including content-based features
X_content = preprocessor.fit_transform(data) # feature vector for al auto
X_content.shape



(29, 105)

In [15]:
feature_price='Sale price'
feature_milleage='Milleage'
feature_Model='Model'
my_modell = "Focus"

#### **Cosinus similarity**
Cosine Similarity misst:

„Wie ähnlich ist Auto i zu Auto j basierend auf ihren Features?“


In [16]:
cosine_sim = cosine_similarity(X_content, X_content)

#cosine_sim
#print(cosine_sim)

In [17]:
car_index = 10
cos_index=cosine_sim[car_index] # similarity scores for the car at index 10
cos_index

array([ 0.05149427,  0.05985091, -0.18662222, -0.01746301,  0.23368532,
       -0.01686178, -0.06010237,  0.15575678,  0.2116303 ,  0.0301061 ,
        1.        ,  0.30766083,  0.50331257,  0.07439678,  0.16615861,
        0.31300882,  0.20719914,  0.18613488,  0.39353984,  0.10489887,
        0.09524015,  0.18426178,  0.19846799,  0.13623594,  0.1759574 ,
       -0.02198392,  0.28021655,  0.18342263,  0.04485375])

#### **Content-Based**
**iloc** wählt Zeilen per Position
(Index des Autos, Ähnlichkeitswert)

In [18]:
size_cosine_sim=len(cosine_sim)
print(size_cosine_sim)

29


In [19]:
data['item_id'] = data.index  # jedes Auto bekommt eine eindeutige ID
#print(data['item_id'].head())
#print(data.loc[3])

len(data)

29

In [20]:
# Function to recommend similar cars based on cosine similarity
def recommend_similar_cars(car_index, top_n=10):
     
    n_items = cosine_sim.shape[0]

    if car_index < 0 or car_index >= n_items:
            raise IndexError(f"car_index must be between 0 and {n_items-1}")
    
    # Get the cosine similarity scores for the item
    similarity_scores = list(enumerate(cosine_sim[car_index]))

    # Sort similar items by similarity score in descending order
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Get the indices of the top N similar items (excluding the first one which is the item itself)
    similar_indices = [i for i, _ in similarity_scores[1:top_n+1]]

    return data.iloc[similar_indices]


#### content-based by model

In [21]:
# Funktion zur Empfehlung ähnlicher Autos basierend auf Cosinus-Ähnlichkeit
def recommend_contentbased_by_model(model_name, top_n=5):
    try:    # Prüfen, ob das Modell im Datensatz existiert
        if model_name not in data[feature_Model].values:
            raise ValueError(f"Modell '{model_name}' nicht im Datensatz gefunden.")

        # Index des Autos anhand des Modellnamens
        car_index = data.index[data[feature_Model] == model_name][0]

        # Cosinus-Ähnlichkeitswerte für das Auto
        similarity_scores = list(enumerate(cosine_sim[car_index]))

        # Sortieren nach Ähnlichkeit absteigend
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

        # Indizes der Top-N ähnlichen Autos (ohne das Auto selbst)
        similar_indices = [i for i, _ in similarity_scores[1:top_n+1]]

        return data.iloc[similar_indices]

    except ValueError as e:
        print(e)
        return pd.DataFrame()



In [22]:
#contentbased_by_model=recommend_contentbased_by_model(model_name, top_n=5)
contentbased_by_model=recommend_contentbased_by_model('Focus')
print(contentbased_by_model)


     Brand               Model  year_of_manufacture  Engine_power(kilowatt)  \
28     Ford    transit connect                 2020                      74   
19     Ford              Focus                 2019                      88   
1   Renault  Grand Scenic BLUE                 2020                      88   
25  Renault            Traffic                 2018                      88   
16     Fiat        Doblo cargo                 2018                      74   

    Engine_power_(Horsepower) Transmission_type  Mileage Fueltype     nextTUV  \
28                        100           manuell   223517   Diesel  2025-08-01   
19                        120           manuell   171803   Diesel  2027-03-01   
1                         120           manuell   154591   Diesel  2027-07-07   
25                        120           manuell   129262   Diesel  2025-04-01   
16                        100           manuell   202662   Diesel  2025-07-07   

    Nber_previous_owners  ... damaged_

In [23]:
def recommend_similar_cars_by_price(price, top_n=10):
    try:
        # Prüfen, ob mindestens ein Auto mit dem Preis existiert
        if price not in data[feature_price].values:
            # Falls exakter Preis nicht existiert, nächstgelegenen Preis nehmen
            closest_price = data[feature_price].iloc[(data[feature_price] - price).abs().argsort()[0]]
            print(f"Kein Auto mit Preis {price} gefunden. Nächstgelegener Preis: {closest_price}")
            price = closest_price

        # Index des Autos mit dem Preis
        car_index = data.index[data[feature_price] == price][0]

        # Cosinus-Ähnlichkeitswerte für das Auto
        similarity_scores = list(enumerate(cosine_sim[car_index]))

        # Sortieren nach Ähnlichkeit absteigend
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

        # Indizes der Top-N ähnlichen Autos (ohne das Auto selbst)
        similar_indices = [i for i, _ in similarity_scores[1:top_n+1]]

        return data.iloc[similar_indices]

    except Exception as e:
        print(f"Fehler: {e}")
        return pd.DataFrame()  # Leere DataFrame zurückgeben


In [24]:
def recommend_contendbased_by_milleage(km, top_n=10):
    try:
        # Prüfen, ob mindestens ein Auto mit dem Kilometerstand existiert
        if km not in data[feature_milleage].values:
            # Falls exakter Kilometerstand nicht existiert, nächstgelegenen Wert nehmen
            closest_km = data[feature_milleage].iloc[(data[feature_milleage] - km).abs().argsort()[0]]
            print(f"Kein Auto mit {km} km gefunden. Nächstgelegener Kilometerstand: {closest_km}")
            km = closest_km

        # Index des Autos mit dem Kilometerstand
        car_index = data.index[data[feature_milleage] == km][0]

        # Cosinus-Ähnlichkeitswerte für das Auto
        similarity_scores = list(enumerate(cosine_sim[car_index]))

        # Sortieren nach Ähnlichkeit absteigend
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

        # Indizes der Top-N ähnlichen Autos (ohne das Auto selbst)
        similar_indices = [i for i, _ in similarity_scores[1:top_n+1]]

        return data.iloc[similar_indices]

    except Exception as e:
        print(f"Fehler: {e}")
        return pd.DataFrame()  # Leere DataFrame zurückgeben


In [25]:
def recommend_contendbased(car_index, top_n=5):
    try:
        recommended_cars = recommend_similar_cars(car_index, top_n=top_n)
        
    except IndexError as e:
        print("Fehler:", e)
        recommended_cars = pd.DataFrame()  # leeres DataFrame als Fallback

    return recommended_cars

In [26]:
recommend_contendbased(5, top_n=5)
#recommend_similar_cars(6, top_n=5)


,Brand,Model,year_of_manufacture,Engine_power(kilowatt),Engine_power_(Horsepower),Transmission_type,Mileage,Fueltype,nextTUV,Nber_previous_owners,...,damaged_rim,damaged_roof_beam,damaged_left_door,damaged_right_door,damaged_seats,interior_dirty,damaged_carosserie,images,Sale price,item_id
20,Ford,Focus,2020,110,150,automatik,178428,Diesel,2026-03-01,2,...,1,1,1,1,1,0,0,A24,4900,20
14,Renault,Kango rapid Blue,2021,70,95,manuell,75832,Diesel,2026-09-01,1,...,0,0,0,0,0,0,0,A22,6400,14
1,Renault,Grand Scenic BLUE,2020,88,120,manuell,154591,Diesel,2027-07-07,1,...,1,0,0,0,0,0,0,A17,6600,1
7,Ford,Fiesta,2020,63,85,manuell,157715,Diesel,2027-07-15,1,...,1,0,0,1,1,0,1,A20,6700,7
17,Renault,Kango rapid Blue,2020,70,95,manuell,68389,Diesel,2026-07-01,1,...,0,0,1,0,0,0,0,A35,6600,17


In [27]:
data.columns = data.columns.str.strip()

#data[['Brand','Model', 'images']]  # funktioniert jetzt


#### **Collaborative Filtering**
Empfiehlt Items basierend auf dem Verhalten vieler Nutzer, nicht auf Item-Features.

Item-Item Cosine Similarity berechnen

#### **Build User_Matrix**

user_item_matrix: Zeilen = Nutzer, Spalten = Autos, Werte = Ratings

In [28]:


data['item_id'] = data.index  # eindeutige ID für jedes Auto

# Simulierte User erstellen
n_users = 5  # number of user
n_items = len(data)

# ratings array
ratings_list = []

np.random.seed(42)  # für Reproduzierbarkeit

#Jeder User bewertet jedes Auto
for user_id in range(1, n_users + 1):
    for item_id in data['item_id']:

        # Zufällige Bewertung, z.B. 1-5
        rating = np.random.randint(1, 6)
        ratings_list.append({
            'user_id': user_id,
            'item_id': item_id,
            'rating': rating
        })

# Ratings DataFrame erstellen
ratings_df = pd.DataFrame(ratings_list)

# User–Item-Matrix erstellen
user_item_matrix = ratings_df.pivot_table(
    index='user_id',      # Zeilen = User
    columns='item_id',    # Spalten = Items
    values='rating',      # Werte = Ratings
    fill_value=0          # keine Bewertung = 0
)




In [29]:
#data['item_id'] # eindeutige ID für jedes Auto
'item_id' in data.columns
type(data['item_id'])# pandas.core.series.Series
data[['item_id']].columns



Index(['item_id'], dtype='object')

In [30]:
ratings_df

,user_id,item_id,rating
0,1,0,4
1,1,1,5
2,1,2,3
3,1,3,5
4,1,4,5
...,...,...,...
140,5,24,1
141,5,25,5
142,5,26,4
143,5,27,4


In [31]:

cosine_sim_item = cosine_similarity(user_item_matrix.T)
cosine_sim_item.shape

(29, 29)

In [32]:
#print(user_item_matrix.head(2))

In [33]:
def recommend_collaborativ_cars(car_index, top_n=5):
    """
    Gibt die Top-N ähnlichen Autos zurück basierend auf Features (Item-Based CF)
    """
    n_items = cosine_sim.shape[0]
    
    if car_index < 0 or car_index >= n_items:
        raise ValueError(f"car_index muss zwischen 0 und {n_items-1} liegen")
    
    # Ähnlichkeiten abrufen
    similarity_scores = list(enumerate(cosine_sim_item[car_index]))
    
    # Sortieren nach Ähnlichkeit (absteigend)
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    # Top-N ähnliche Autos, eigenes Auto ausschließen
    similar_indices = [i for i, _ in similarity_scores[1:top_n+1]]
    
    # DataFrame zurückgeben
    return data.iloc[similar_indices]


In [34]:
# Index des Autos, das empfohlen werden soll
car_index = 1  # z.B. erstes Auto in df

recommended= recommend_collaborativ_cars(car_index, top_n=5)
#print(recommended[['Brand','Model','year of manufacture']])
print(recommended)



      Brand                  Model  year_of_manufacture  \
28     Ford        transit connect                 2020   
9   Renault      Grand Scenic BLUE                 2018   
3      Ford                  Focus                 2020   
21  Hyundai  Nexo Fuel Cell Sports                 2022   
19     Ford                  Focus                 2019   

    Engine_power(kilowatt)  Engine_power_(Horsepower) Transmission_type  \
28                      74                        100           manuell   
9                       89                        120           manuell   
3                       88                        120           manuell   
21                     120                        163         automatik   
19                      88                        120           manuell   

    Mileage  Fueltype     nextTUV  Nber_previous_owners  ... damaged_rim  \
28   223517    Diesel  2025-08-01                     1  ...           1   
9    123181    Diesel  2027-02-01         

#### Hybrid Filtering

In [35]:
def recommend_hybrid(car_index, top_n=5):
    """
    Hybrid Recommendation:
    alpha = Gewicht für Collaborative Filtering (0-1)
    (1-alpha) = Gewicht für Content-Based
    """
    # Content-Based Score
    contentFiltering_scores = recommend_contendbased(car_index, top_n=top_n)
    
    # CF Score
    collaborative_scores = recommend_collaborativ_cars(car_index, top_n=top_n)

   # merge scores
    hybrid_rec = pd.concat(
      [
        contentFiltering_scores,
        collaborative_scores]
       
    ).drop_duplicates()
    
    return hybrid_rec.head(top_n)


In [36]:
recommend_hybrid(0, top_n=5)

,Brand,Model,year_of_manufacture,Engine_power(kilowatt),Engine_power_(Horsepower),Transmission_type,Mileage,Fueltype,nextTUV,Nber_previous_owners,...,damaged_rim,damaged_roof_beam,damaged_left_door,damaged_right_door,damaged_seats,interior_dirty,damaged_carosserie,images,Sale price,item_id
21,Hyundai,Nexo Fuel Cell Sports,2022,120,163,automatik,36640,Hydrogen,2027-05-01,2,...,0,0,0,0,1,1,0,A31,10000,21
13,Hyundai,loniq5,2021,218,163,automatik,87012,Electronic,2026-09-01,2,...,0,0,0,0,0,1,0,A28,17100,13
2,Hyundai,Nexo Fuel Cell Sports,2024,120,163,automatik,37180,Hydrogen,2025-11-01,1,...,1,0,1,1,0,1,0,A9,8600,2
22,Hyundai,Nexo Fuel Cell Sports,2021,120,163,Automatik,19061,Hydrogen,2027-07-07,2,...,0,1,0,0,0,1,1,A11,8100,22
18,Renault,Grand Scenic BLUE,2020,88,120,manuell,19457,Diesel,2027-08-01,2,...,1,0,0,0,1,1,1,A4,6200,18


In [37]:
data['images'] = data['images'].str.strip().str.lstrip('/')
(data['images'].head(5))


0    A25
1    A17
2     A9
3    A16
4    A13
Name: images, dtype: object

In [38]:
from pathlib import Path
import streamlit as st



BASE_DIR = Path("../Data/usecar_image/")

def load_all_car_images():

   # folder_path = BASE_DIR / folder_name

    car_images = {}

    for car_folder in BASE_DIR.iterdir():
        if car_folder.is_dir():
            car_id = car_folder.name  # z.B. A1
            
            images = [
                str(img)
                for img in car_folder.glob("*")
                if img.suffix.lower() in ['.png', '.jpg', '.jpeg']
            ]

            car_images[car_id] = images

    return car_images


car_images_dict = load_all_car_images()
#data['image_folder'] = data['images'].str.extract(r'(A\d+)')
#data['image_folder'] = data['images'].apply(lambda x: Path(x[0]).parts[-2] if x else None)

#data['image_files'] = data['image_folder'].map(car_images_dict)
#data['image_files'].head(5)
#print(data['images'].head())

# # --- Streamlit UI ---
# st.title("Car Gallery with Features")

# # Interaktive Auswahl: Auto auswählen
# selected_car = st.selectbox("Wähle ein Auto", data['image_folder'].dropna().unique())

# row = data[data['image_folder'] == selected_car].iloc[0]

# # Auto-Features anzeigen
# st.subheader(f"{row['image_folder']} - {row['Model']} ({row['Brand']})")

# Bilder anzeigen
# if row['images']:
#     for img_path in row['images']:
#         st.image(img_path, width=300)
# else:
#     st.write("Keine Bilder vorhanden")

# for car, images in car_images_dict.items():
#     print(f"\nAuto: {car}")
#     for img in images:
#         print("  ", img)




In [39]:
# data['image_files']
# for idx, row in data.iterrows():
#     print("\nAuto:", row['Brand'], row['Model'])
#     print("Bilder:", row['image_files'])



In [40]:
BASE_DIR = Path("../Data/usecar_image")

def get_images(folder_name):
    folder_path = BASE_DIR / folder_name
    print("Checking:", folder_path)  # DEBUG

    if not folder_path.exists():
        return []
    
    return [
        str(img)
        for img in folder_path.glob("*")
        if img.suffix.lower() in ['.png', '.jpg', '.jpeg']
    ]

#data['image_files'] = data['images'].apply(get_images)



In [41]:
import os

# Funktion, um alle Bilder eines Ordners zu bekommen
def list_images(folder_path):
    # Absoluter Pfad, wenn nötig
    folder_path = folder_path.strip()  # Leerzeichen entfernen
    if not os.path.exists(folder_path):
        return []  # falls Ordner fehlt
    # Nur Bilder mit gängigen Endungen
    images = [os.path.join(folder_path, f) for f in os.listdir(folder_path)
              if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    return images

#floder=list_images(data['images'][0])
(data['images'].head(5))
# Neue Spalte "image_files" mit Liste aller Bilder
data['image_files'] = data['images'].apply(list_images)

#print(os.getcwd())

# Beispiel ausgeben
for idx, row in data.iterrows():
     print(f"Car: {row['Brand']} {row['Model']}, Bilder: {row['image_files']}")


Car: Hyundai Nexo Fuel Cell Sports, Bilder: []
Car: Renault Grand Scenic BLUE, Bilder: []
Car: Hyundai Nexo Fuel Cell Sports, Bilder: []
Car: Ford Focus, Bilder: []
Car: Renault Kango rapid Blue, Bilder: []
Car: Ford Focus, Bilder: []
Car: Renault Grand Scenic BLUE, Bilder: []
Car: Ford Fiesta, Bilder: []
Car: Renault Grand Scenic BLUE, Bilder: []
Car: Renault Grand Scenic BLUE, Bilder: []
Car: Renault Grand Scenic BLUE, Bilder: []
Car: Renault Kango rapid Blue, Bilder: []
Car: Renault Grand Scenic BLUE, Bilder: []
Car: Hyundai loniq5, Bilder: []
Car: Renault Kango rapid Blue, Bilder: []
Car: Renault Grand Scenic BLUE, Bilder: []
Car: Fiat Doblo cargo, Bilder: []
Car: Renault Kango rapid Blue, Bilder: []
Car: Renault Grand Scenic BLUE, Bilder: []
Car: Ford Focus, Bilder: []
Car: Ford Focus, Bilder: []
Car: Hyundai Nexo Fuel Cell Sports, Bilder: []
Car: Hyundai Nexo Fuel Cell Sports, Bilder: []
Car: Renault Grand Scenic BLUE, Bilder: []
Car: Renault Grand Scenic BLUE, Bilder: []
Car: Re

#### Save Variable and function

In [42]:
#dump(my_modell, "my_model.joblib")
dump({"model": my_modell, "k": 10}, "config.joblib")



['config.joblib']

In [43]:
dump( recommend_contentbased_by_model(my_modell), "contentbased_by_mode.joblib")


['contentbased_by_mode.joblib']

In [44]:
dump( data, "dataFrame.joblib")


['dataFrame.joblib']

In [45]:
dump( cosine_sim, "cosine_similarity.joblib")



['cosine_similarity.joblib']